In [ ]:
# 1. Install Advanced Dependencies
!pip install pygeohash catboost xgboost scikit-learn category_encoders -q

import pandas as pd
import numpy as np
import pygeohash as pgh
import warnings
from catboost import CatBoostRegressor, Pool
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import BayesianRidge
from category_encoders import TargetEncoder

warnings.filterwarnings('ignore')

print("=== Phase 1: Data Ingestion & Target Transformation ===")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
test_index = test['Index'].copy()

# Combine datasets
train['is_train'] = 1
test['is_train'] = 0
test['demand'] = 0
df = pd.concat([train, test], axis=0).reset_index(drop=True)

# RESEARCH METHOD: Target Transformation to stabilize gradients
train['demand_log'] = np.log1p(train['demand'])

print("=== Phase 2: Deep Spatial & Temporal Synthesization ===")

# A. Spatial Extraction
df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])

# B. Distance to Geographic Center (Haversine Approximation)
center_lat, center_lon = df['lat'].median(), df['lon'].median()
df['dist_to_center'] = np.sqrt((df['lat'] - center_lat)**2 + (df['lon'] - center_lon)**2)

# C. Spatial PCA (Learning Diagonal Traffic Corridors)
print("--> Extracting Spatial Principal Components...")
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(df[['lat', 'lon']])
df['pca_coord_1'] = pca_coords[:, 0]
df['pca_coord_2'] = pca_coords[:, 1]

# D. Temporal Extraction & Cyclical Encoding
df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
df['time_in_mins'] = df['hour'] * 60 + df['minute']

print("--> Applying Trigonometric Temporal Encodings...")
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)

# E. Imputation & Base Categoricals
df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda x: x.fillna(x.median()))
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
df['RoadType'] = df['RoadType'].fillna('Unknown')
df['Weather'] = df['Weather'].fillna('Unknown')

# Feature Interactions
df['temp_x_time'] = df['Temperature'] * df['time_in_mins']

# F. Prepare Data Split
drop_cols = ['Index', 'timestamp', 'hour', 'minute', 'is_train']
train_features = df[df['is_train'] == 1].drop(columns=drop_cols + ['demand'])
test_features = df[df['is_train'] == 0].drop(columns=drop_cols + ['demand'])

y_log = train['demand_log'] # We train on the Log-Transformed Target
y_true = train['demand']    # We evaluate on the True Target

# Define Categorical Columns
cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

print("\n=== Phase 3: Level-1 Base Model Training (Out-of-Fold) ===")
# Setup K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Arrays to store Level-1 Meta-Features for the Level-2 Stacker
oof_cb = np.zeros(len(train_features))
oof_xgb = np.zeros(len(train_features))
test_cb_preds = np.zeros((len(test_features), 5))
test_xgb_preds = np.zeros((len(test_features), 5))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_features, y_log)):
    print(f"\n--- FOLD {fold + 1}/5 ---")

    # 1. Data Split
    X_tr, X_va = train_features.iloc[train_idx].copy(), train_features.iloc[val_idx].copy()
    y_tr, y_va = y_log.iloc[train_idx], y_log.iloc[val_idx]
    X_test_fold = test_features.copy()

    # 2. Target Encoding (Smooth Bayesian Mean Encoding for XGBoost)
    # We apply this INSIDE the fold to entirely prevent data leakage
    encoder = TargetEncoder(cols=cat_cols, smoothing=10)
    X_tr_xgb = encoder.fit_transform(X_tr, y_tr)
    X_va_xgb = encoder.transform(X_va)
    X_test_xgb = encoder.transform(X_test_fold)

    # CatBoost handles strings natively, so we cast them
    X_tr_cb, X_va_cb, X_test_cb = X_tr.copy(), X_va.copy(), X_test_fold.copy()
    for col in cat_cols:
        X_tr_cb[col] = X_tr_cb[col].astype(str)
        X_va_cb[col] = X_va_cb[col].astype(str)
        X_test_cb[col] = X_test_cb[col].astype(str)

    train_pool = Pool(X_tr_cb, y_tr, cat_features=cat_cols)
    val_pool = Pool(X_va_cb, y_va, cat_features=cat_cols)
    test_pool = Pool(X_test_cb, cat_features=cat_cols)

    # ------------------ Train CatBoost (GPU) ------------------
    cb_model = CatBoostRegressor(
        iterations=3500, learning_rate=0.03, depth=8, l2_leaf_reg=5,
        eval_metric='RMSE', task_type='GPU', random_seed=42+fold,
        early_stopping_rounds=250, verbose=0
    )
    cb_model.fit(train_pool, eval_set=val_pool)

    # Predict and Reverse Log1p transform (expm1)
    oof_cb[val_idx] = np.expm1(cb_model.predict(val_pool))
    test_cb_preds[:, fold] = np.expm1(cb_model.predict(test_pool))

    # ------------------ Train XGBoost (CUDA) ------------------
    xgb_model = xgb.XGBRegressor(
        n_estimators=3500, learning_rate=0.03, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, tree_method='hist', device='cuda',
        random_state=42+fold, early_stopping_rounds=250, reg_lambda=5
    )
    xgb_model.fit(X_tr_xgb, y_tr, eval_set=[(X_va_xgb, y_va)], verbose=0)

    # Predict and Reverse Log1p transform
    oof_xgb[val_idx] = np.expm1(xgb_model.predict(X_va_xgb))
    test_xgb_preds[:, fold] = np.expm1(xgb_model.predict(X_test_xgb))

    # Fold Evaluation on True Target
    print(f"CatBoost Fold R2: {r2_score(y_true.iloc[val_idx], oof_cb[val_idx]):.4f}")
    print(f"XGBoost Fold R2:  {r2_score(y_true.iloc[val_idx], oof_xgb[val_idx]):.4f}")

print("\n=== Phase 4: Level-2 Meta-Learner (Stacking) ===")
# We take the Out-Of-Fold predictions from our base models and use them as
# features to train a Meta-Model. The Meta-Model learns how to best weight them dynamically.

meta_X_train = np.column_stack((oof_cb, oof_xgb))
meta_y_train = y_true # Train meta-model on actual true demand

print("--> Training Bayesian Ridge Meta-Learner...")
meta_model = BayesianRidge()
meta_model.fit(meta_X_train, meta_y_train)

# Evaluate the Meta-Learner
final_oof_preds = meta_model.predict(meta_X_train)
stack_r2 = r2_score(y_true, final_oof_preds)
stack_score = max(0, 100 * stack_r2)
print(f"\n--> TOTAL LEVEL-2 STACKING R2 SCORE: {stack_r2:.4f}")
print(f"--> ESTIMATED LEADERBOARD SCORE: {stack_score:.2f}")

print("\n=== Phase 5: Generating Final Predictions ===")
# Average the test predictions across all 5 folds for both base models
final_test_cb = test_cb_preds.mean(axis=1)
final_test_xgb = test_xgb_preds.mean(axis=1)

# Pass the averaged base predictions into the Meta-Learner
meta_X_test = np.column_stack((final_test_cb, final_test_xgb))
final_submission_preds = meta_model.predict(meta_X_test)

# Ensure no negative demand predictions (enforcing physical reality)
final_submission_preds = np.clip(final_submission_preds, 0, None)

submission = pd.DataFrame({
    'Index': test_index,
    'demand': final_submission_preds
})

submission.to_csv('research_stacking_submission.csv', index=False)
print("Success! 'research_stacking_submission.csv' generated.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 7.9 MB/s eta 0:00:00
=== Phase 1: Data Ingestion & Target Transformation ===
=== Phase 2: Deep Spatial & Temporal Synthesization ===
--> Extracting Spatial Principal Components...
--> Applying Trigonometric Temporal Encodings...

=== Phase 3: Level-1 Base Model Training (Out-of-Fold) ===

--- FOLD 1/5 ---
CatBoost Fold R2: 0.9537
XGBoost Fold R2:  0.9599

--- FOLD 2/5 ---
CatBoost Fold R2: 0.9538
XGBoost Fold R2:  0.9593

--- FOLD 3/5 ---
CatBoost Fold R2: 0.9539
XGBoost Fold R2:  0.9596

--- FOLD 4/5 ---
CatBoost Fold R2: 0.9489
XGBoost Fold R2:  0.9542

--- FOLD 5/5 ---
CatBoost Fold R2: 0.9539
XGBoost Fold R2:  0.9600

=== Phase 4: Level-2 Meta-Learner (Stacking) ===
--> Training Bayesian Ridge Meta-Learner...

--> TOTAL LEVEL-2 STACKING R2 SCORE: 0.9589
--> ESTIMATED LEADERBOARD SCORE: 95.89

=== Phase 5: Generating Final Predictions ===
Success! 'research_stacking_submission.csv' generated.
